# Week 5 Day 3: LangGraph — Stateful, Multi-Step & Cyclical Agent Workflows

This notebook acts as the complete deliverable containing executable solutions for Tasks 1 through 5.

In [5]:
# Install necessary dependencies for this notebook
# NOTE: Run this cell first! If you still see red squiggly lines, restart your Jupyter Kernel.
%pip install -qU langgraph langgraph-checkpoint

Note: you may need to restart the kernel to use updated packages.


## Task 1: Graph Concepts & State Design

### Core Building Blocks
- **State Object:** A shared data structure (like `TypedDict`) passed from node to node. Nodes return updates that LangGraph merges into this state.
- **StateGraph:** The orchestrator that manages nodes, edges, and execution based on the State schema.
- **Nodes:** Python functions or Runnables that perform the actual work (calling an LLM, searching, etc.).
- **Edges:** Fixed deterministic transitions between nodes (e.g., A always goes to B).
- **Conditional Edges:** Dynamic routing paths where a function inspects the current state and decides the next node, enabling cyclical loops.

### Workflow Diagram (Research Assistant)
```mermaid
graph TD
    START((START)) --> SearchNode[Search Web]
    SearchNode --> DraftNode[Draft Answer]
    DraftNode --> CritiqueNode[Critique Draft]
    CritiqueNode --> CheckCondition{Is Approved?}
    CheckCondition -->|No (Needs Revision)| SearchNode
    CheckCondition -->|Yes (Or Max Revisions)| END((END))
```

In [1]:
from typing import List, Annotated, TypedDict
import operator

# Task 1: State Design for a Research Assistant
class ResearchState(TypedDict):
    query: str
    # operator.add ensures new notes are appended rather than overwritten
    research_notes: Annotated[List[str], operator.add]
    draft: str
    critique: str
    revision_count: int

---
## Task 2: Build a Linear Graph
Implementing a simple 4-node linear graph: `plan -> retrieve -> generate -> format`.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class LinearState(TypedDict):
    input_query: str
    plan: str
    retrieved_docs: list[str]
    draft: str
    final_response: str

def plan_step(state: LinearState):
    return {"plan": f"Plan: 1. Search DB, 2. Synthesize for '{state.get('input_query')}'"}

def retrieve_step(state: LinearState):
    return {"retrieved_docs": ["Doc A", "Doc B"]}

def generate_step(state: LinearState):
    return {"draft": f"Draft based on {len(state.get('retrieved_docs', []))} docs."}

def format_step(state: LinearState):
    return {"final_response": f"### Final Output ###\n{state.get('draft', '')}"}

linear_workflow = StateGraph(LinearState)
linear_workflow.add_node("plan", plan_step)
linear_workflow.add_node("retrieve", retrieve_step)
linear_workflow.add_node("generate", generate_step)
linear_workflow.add_node("format", format_step)

linear_workflow.add_edge(START, "plan")
linear_workflow.add_edge("plan", "retrieve")
linear_workflow.add_edge("retrieve", "generate")
linear_workflow.add_edge("generate", "format")
linear_workflow.add_edge("format", END)

linear_app = linear_workflow.compile()

print("--- Executing Linear Graph ---")
for step in linear_app.stream({"input_query": "Explain LangGraph"}):
    for node, update in step.items():
        print(f"[{node}]: {update}")

--- Executing Linear Graph ---
[plan]: {'plan': "Plan: 1. Search DB, 2. Synthesize for 'Explain LangGraph'"}
[retrieve]: {'retrieved_docs': ['Doc A', 'Doc B']}
[generate]: {'draft': 'Draft based on 2 docs.'}
[format]: {'final_response': '### Final Output ###\nDraft based on 2 docs.'}


---
## Task 3: Add Conditional Edges & Cycles
Introducing a `critique` node that routes back to `generate` if quality is low, or forward to `format` if acceptable.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class CyclicState(TypedDict):
    draft: str
    critique: str
    revision_count: int
    final_response: str

def generate_cyclic(state: CyclicState):
    rev = state.get("revision_count", 0)
    return {"draft": f"Draft v{rev + 1}"}

def critique_cyclic(state: CyclicState):
    rev = state.get("revision_count", 0)
    # Mock critique: pass on 3rd attempt
    if rev < 2:
        critique = "Needs more detail."
    else:
        critique = "Acceptable."
    return {"critique": critique, "revision_count": rev + 1}

def route_critique(state: CyclicState):
    rev = state.get("revision_count", 0)
    if rev >= 3 or "Acceptable" in state.get("critique", ""):
        return "format"
    return "generate"

def format_cyclic(state: CyclicState):
    return {"final_response": f"Final: {state.get('draft')} | Critique: {state.get('critique')}"}

cyclic_workflow = StateGraph(CyclicState)
cyclic_workflow.add_node("generate", generate_cyclic)
cyclic_workflow.add_node("critique", critique_cyclic)
cyclic_workflow.add_node("format", format_cyclic)

cyclic_workflow.add_edge(START, "generate")
cyclic_workflow.add_edge("generate", "critique")
cyclic_workflow.add_conditional_edges("critique", route_critique, {"generate": "generate", "format": "format"})
cyclic_workflow.add_edge("format", END)

cyclic_app = cyclic_workflow.compile()

print("--- Executing Cyclic Graph ---")
for step in cyclic_app.stream({"revision_count": 0}):
    for node, update in step.items():
        print(f"[{node}]: {update}")

--- Executing Cyclic Graph ---
[generate]: {'draft': 'Draft v1'}
[critique]: {'critique': 'Needs more detail.', 'revision_count': 1}
[generate]: {'draft': 'Draft v2'}
[critique]: {'critique': 'Needs more detail.', 'revision_count': 2}
[generate]: {'draft': 'Draft v3'}
[critique]: {'critique': 'Acceptable.', 'revision_count': 3}
[format]: {'final_response': 'Final: Draft v3 | Critique: Acceptable.'}


---
## Task 4: Human-in-the-Loop & Interrupts
Pausing execution before a risky action to await human approval.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class HITLState(TypedDict):
    task: str
    human_approved: bool
    result: str

def plan_action(state: HITLState):
    return {}

def execute_action(state: HITLState):
    if state.get("human_approved"):
        return {"result": "SUCCESS: Action executed!"}
    return {"result": "ABORTED: Action rejected!"}

hitl_workflow = StateGraph(HITLState)
hitl_workflow.add_node("plan", plan_action)
hitl_workflow.add_node("execute", execute_action)
hitl_workflow.add_edge(START, "plan")
hitl_workflow.add_edge("plan", "execute")
hitl_workflow.add_edge("execute", END)

memory_hitl = MemorySaver()
hitl_app = hitl_workflow.compile(checkpointer=memory_hitl, interrupt_before=["execute"])

config = {"configurable": {"thread_id": "transaction_1"}}
print("--- 1. Starting execution... ---")
for step in hitl_app.stream({"task": "Send $500", "human_approved": False}, config):
    pass

print("\n--- 2. Graph paused! Waiting for human input... ---")
print("Current state:", hitl_app.get_state(config).values)

print("\n--- 3. Simulating Human Approval & Resuming... ---")
hitl_app.update_state(config, {"human_approved": True})
for step in hitl_app.stream(None, config):
    for node, update in step.items():
        print(f"[{node}]: {update}")

--- 1. Starting execution... ---

--- 2. Graph paused! Waiting for human input... ---
Current state: {'task': 'Send $500', 'human_approved': False}

--- 3. Simulating Human Approval & Resuming... ---
[execute]: {'result': 'SUCCESS: Action executed!'}


### Discussion: When to use Human-in-the-Loop vs Full Autonomy
**Human-in-the-loop** is required for irreversible actions, financial transactions, destructive operations (like modifying databases), or external communications (sending emails to clients). 

**Full Autonomy** is acceptable for internal research, data formatting, internal report generation, or read-only workflows where a hallucination only results in an incorrect document rather than real-world damage.

---
## Task 5: Persistence & Debugging
Demonstrating state history tracking and time-travel debugging.

In [4]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class ChatState(TypedDict):
    messages: Annotated[list[str], operator.add]

def chat_node(state: ChatState):
    return {"messages": [f"Echo: {state['messages'][-1]}"]}

chat_workflow = StateGraph(ChatState)
chat_workflow.add_node("chat", chat_node)
chat_workflow.add_edge(START, "chat")
chat_workflow.add_edge("chat", END)

chat_app = chat_workflow.compile(checkpointer=MemorySaver())
chat_config = {"configurable": {"thread_id": "session_abc"}}

print("--- Turn 1 ---")
chat_app.invoke({"messages": ["Hi"]}, chat_config)
print("--- Turn 2 ---")
chat_app.invoke({"messages": ["Help me"]}, chat_config)

print("\n--- State History ---")
history = list(chat_app.get_state_history(chat_config))
for idx, state in enumerate(history):
    print(f"[{idx}] Checkpoint ID {state.config['configurable']['checkpoint_id'][:8]}... : {state.values.get('messages')}")

print("\n--- Time Travel (Branching from Turn 1) ---")
turn_1_state = next(s for s in history if len(s.values.get("messages", [])) == 2)
chat_app.invoke({"messages": ["Tell a joke instead"]}, turn_1_state.config)

print("\n--- New Branched State ---")
print(chat_app.get_state(chat_config).values)

--- Turn 1 ---
--- Turn 2 ---

--- State History ---
[0] Checkpoint ID 1f185c51... : ['Hi', 'Echo: Hi', 'Help me', 'Echo: Help me']
[1] Checkpoint ID 1f185c51... : ['Hi', 'Echo: Hi', 'Help me']
[2] Checkpoint ID 1f185c51... : ['Hi', 'Echo: Hi']
[3] Checkpoint ID 1f185c51... : ['Hi', 'Echo: Hi']
[4] Checkpoint ID 1f185c51... : ['Hi']
[5] Checkpoint ID 1f185c51... : []

--- Time Travel (Branching from Turn 1) ---

--- New Branched State ---
{'messages': ['Hi', 'Echo: Hi', 'Tell a joke instead', 'Echo: Tell a joke instead']}


### Comparison: LangChain AgentExecutor vs. LangGraph
- **AgentExecutor:** Best for simple ReAct Q&A agents that just need a list of tools and autonomy to loop until they find an answer. Hard to strictly control the flow.
- **LangGraph:** Best for production workflows where you need state persistence, guaranteed sequential execution, human-in-the-loop approvals, or complex multi-agent architectures.